# Web Scraping--SINA and Xeno-Canto

Gathers everything the rest of the pipeline runs on. Crickets and katydids
come from [SINA](https://orthsoc.org/sina/)--one page per species, each with
a spectrogram, an audio file, a description, and a range map. Frogs come from
the [Xeno-Canto](https://xeno-canto.org) API instead, which hands back clean
JSON, so there's no HTML to parse.

Produces three DataFrames (`cricket_df`, `katydid_df`, `frog_df`) shared with
the processing notebooks via `%store`, then downloads every referenced
spectrogram, audio file, and map into `~/Discrete_Signals/`.

**Run order:** this notebook first, then whichever processing notebooks you
want. You'll need a free Xeno-Canto API key in `XENO_CANTO_API_KEY` for the
frog pull.

## Setup

In [ ]:
import os
import random
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [ ]:
# Show full column contents and every row when a DataFrame is displayed.
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

DATA_DIR = Path('data')            # small lookup files kept next to the notebooks
DISCRETE_SIGNALS_DIR = Path.home() / 'Discrete_Signals'   # downloaded media

## Scraping SINA

Crickets and katydids share the same site layout, so one set of helpers
handles both--only the starting list page differs.

In [ ]:
SINA_BASE_URL = 'https://orthsoc.org/sina/'

# SINA asks scrapers to identify themselves rather than impersonate a browser.
REQUEST_HEADERS = {
    'User-Agent': (
        'AcademicResearchBot '
        '(non-commercial communication-signals research project)'
    ),
    'Accept': 'text/html,application/xhtml+xml',
    'Accept-Language': 'en-US,en;q=0.9',
    'Connection': 'keep-alive',
}

# Words that mark an <img> on a recording block as the spectrogram/waveform
# figure (SINA is inconsistent about which term it uses).
SPECTROGRAM_KEYWORDS = ('spectrogram', 'sonogram', 'waveform', 'graph', 'graphed song')

In [ ]:
def make_session():
    """A requests session preloaded with the SINA-friendly headers."""
    session = requests.Session()
    session.headers.update(REQUEST_HEADERS)
    return session


def fetch_soup(session, url):
    """GET a page and return it parsed, letting requests guess the encoding."""
    response = session.get(url, timeout=15)
    response.encoding = response.apparent_encoding
    return BeautifulSoup(response.text, 'html.parser')


def get_species_list(list_url, session):
    """Parse a SINA list page (cricklist.htm / katylist.htm) into
    ``[{"species": name, "url": species_page_url}, ...]``."""
    soup = fetch_soup(session, list_url)
    species = []
    for heading in soup.find_all('h3', class_='species'):
        link = heading.find('a')
        if link:
            species.append({
                'species': link.get_text(strip=True),
                'url': urljoin(SINA_BASE_URL, link['href']),
            })
    return species

In [ ]:
def _map_gif_from_href(href, species_url):
    """SINA links range maps as ``<code>m.htm`` detail pages; the full-size GIF
    lives at ``<code>mc.gif`` in the same folder. Returns the GIF URL or None."""
    # We never fetch the m.htm page itself -- the numeric code in its name is
    # all we need to build the direct GIF link.
    match = re.search(r'(\d+)m\.htm', href, re.I)
    if not match:
        return None
    return urljoin(species_url, f'{match.group(1)}mc.gif')


def find_range_map_url(soup, species_url):
    """Locate a species page's range-map GIF.

    Most pages put it in a ``table.images`` block. A few older pages use a
    plain caption row instead, with the map image in the row above the cell
    whose caption says "map"--handled as a fallback.
    """
    image_table = soup.find('table', class_='images')
    if image_table:
        link = image_table.find('a', href=True)
        if link:
            map_url = _map_gif_from_href(link['href'], species_url)
            if map_url:
                return map_url

    # Fallback layout: a caption cell reading "map" with the actual image in the
    # matching column of the row directly above it. Column position matters--# these rows can hold several figures side by side.
    for row in soup.find_all('tr'):
        caption_cells = row.find_all('td', class_='captions')
        for column_index, caption_cell in enumerate(caption_cells):
            if 'map' not in caption_cell.get_text(' ', strip=True).lower():
                continue
            previous_row = row.find_previous_sibling('tr')
            if not previous_row:
                continue
            image_cells = previous_row.find_all('td', class_='images')
            if column_index >= len(image_cells):
                continue
            link = image_cells[column_index].find('a', href=True)
            if link:
                map_url = _map_gif_from_href(link['href'], species_url)
                if map_url:
                    return map_url
    return None

In [ ]:
def parse_temperature(description_text):
    """Pull the recording temperature (as a string of digits/decimal point) out
    of a description. SINA writes it just before the degree sign, e.g.
    "...calling at 24.5°C from..."--so take the few characters before "°"
    and strip everything that isn't part of a number."""
    if '°' not in description_text:
        return None
    degree_index = description_text.find('°')
    # Grab a short window before the sign and peel off everything that isn't a
    # number. "Â" shows up when the page's degree sign gets mis-decoded.
    candidate = description_text[max(0, degree_index - 5):degree_index]
    for junk in ('Â', ':', ';', ',', ' '):
        candidate = candidate.replace(junk, '')
    candidate = candidate.strip('abcdefghijklmnopqrstuvwxyz').strip('.')
    return candidate or None


def parse_location(description_text):
    """Pull the recording location out of a description. It usually follows
    "from" or "in" and runs up to the temperature marker (or end of text)."""
    # Deliberately loose: "in" matches inside other words too, so this is only a
    # first pass--review_locations (below) is where the values actually get
    # cleaned up by hand.
    for keyword in ('from', 'in'):
        if keyword in description_text:
            start = description_text.find(keyword) + len(keyword) + 1
            end = description_text.find('°') if '°' in description_text else len(description_text)
            return description_text[start:end].strip(' ,;.Â')
    return 'No location found'

In [ ]:
def extract_recordings(recording_soup, species_name, species_url, map_url):
    """Turn one species page's recording blocks into a list of row dicts.

    A ``recordingnopadding`` block is an extra clip of the recording above it, so
    it inherits that recording's description (which carries the temperature and
    location). Blocks with no spectrogram figure are skipped.
    """
    rows = []
    recording_blocks = recording_soup.find_all(
        'div', class_=lambda value: value and 'recording' in value
    )
    for block in recording_blocks:
        description_text = block.get_text(' ', strip=True)
        if 'recordingnopadding' in block.get('class', []):
            # This block is a clip of the one above it and has no description of
            # its own; borrow the parent's for temperature/location.
            parent_block = block.find_previous('div', class_='recording')
            if parent_block:
                description_text = parent_block.get_text(' ', strip=True)

        audio_url = None
        audio_tag = block.find('audio')
        if audio_tag:
            source_tag = audio_tag.find('source', src=True)
            if source_tag:
                audio_url = urljoin(species_url, source_tag['src'])

        spectrogram_url = None
        for image in block.find_all('img', src=True):
            # Some blocks only identify the figure in the surrounding prose, not
            # the src or alt text, so search all three together.
            haystack = f"{image['src'].lower()} {(image.get('alt') or '').lower()} {description_text.lower()}"
            if any(keyword in haystack for keyword in SPECTROGRAM_KEYWORDS):
                spectrogram_url = urljoin(species_url, image['src'])
                break
        if not spectrogram_url:
            continue

        rows.append({
            'Species': species_name,
            'URL': species_url,
            'Spectrogram': spectrogram_url,
            'Audio': audio_url,
            'Description': description_text,
            'Temperature': parse_temperature(description_text),
            'Location': parse_location(description_text),
            'Map': map_url,
        })
    return rows

In [ ]:
def scrape_sina(list_url):
    """Scrape every species linked from a SINA list page into a DataFrame with
    one row per recording. Pauses 1-2 s between species to stay polite."""
    session = make_session()
    species_list = get_species_list(list_url, session)

    all_rows = []
    for species in species_list:
        page_soup = fetch_soup(session, species['url'])
        map_url = find_range_map_url(page_soup, species['url'])
        all_rows.extend(
            extract_recordings(page_soup, species['species'], species['url'], map_url)
        )
        time.sleep(random.uniform(1, 2))

    return pd.DataFrame(all_rows)

## Manual location review

Descriptions are inconsistent enough that `parse_location` gets close but not
always clean. `review_locations` lets you walk through the parsed values and
correct them by hand. Running it is optional--the last pass's corrections are
saved in `data/*_locations_reviewed.csv` and applied below, positionally,
which is why the scrape order has to stay stable.

In [ ]:
def review_locations(location_series):
    """Interactively confirm/correct a column of parsed locations.
    Returns a new list the same length as the input."""
    reviewed = []
    for current in location_series:
        print(f'Current location: {current}')
        replacement = input('Replacement (Enter to keep): ').strip()
        reviewed.append(replacement or current)
    print('Done.')
    return reviewed


def apply_reviewed_locations(df, reviewed_csv):
    """Overwrite ``df['Location']`` with the hand-reviewed values in
    ``reviewed_csv`` (one location per line, in scrape order)."""
    reviewed = pd.read_csv(reviewed_csv)['Location'].tolist()
    if len(reviewed) != len(df):
        raise ValueError(
            f'{reviewed_csv.name} has {len(reviewed)} rows but the scrape '
            f'produced {len(df)}--re-run review_locations to regenerate it.'
        )
    df = df.copy()
    df['Location'] = reviewed
    return df

## Crickets

In [ ]:
cricket_df = scrape_sina('https://orthsoc.org/sina/cricklist.htm')

In [ ]:
# Rename to the column names the cricket processing notebook expects.
cricket_df.columns = [
    'Species', 'URL', 'Spectrogram', 'Audio_Link',
    'Description of Whole Audio File', 'Temperature (°C)', 'Location', 'Map',
]

In [ ]:
# cricket_reviewed_locations = review_locations(cricket_df["Location"])
# pd.DataFrame({"Location": cricket_reviewed_locations}).to_csv(
#     DATA_DIR / "cricket_locations_reviewed.csv", index=False)

cricket_df = apply_reviewed_locations(cricket_df, DATA_DIR / 'cricket_locations_reviewed.csv')

In [ ]:
%store cricket_df
cricket_df

## Katydids

In [ ]:
katydid_df = scrape_sina('https://orthsoc.org/sina/katylist.htm')

In [ ]:
# Rename to the column names the katydid processing notebook expects. Note the
# description column name differs from the cricket one ("Description", not
# "Description of Whole Audio File")--the processing notebooks rely on this.
katydid_df.columns = [
    'Species', 'URL', 'Spectrogram', 'Audio_Link',
    'Description', 'Temperature (°C)', 'Location', 'Map',
]

In [ ]:
# katydid_reviewed_locations = review_locations(katydid_df["Location"])
# pd.DataFrame({"Location": katydid_reviewed_locations}).to_csv(
#     DATA_DIR / "katydid_locations_reviewed.csv", index=False)

katydid_df = apply_reviewed_locations(katydid_df, DATA_DIR / 'katydid_locations_reviewed.csv')

In [ ]:
%store katydid_df
katydid_df

## Frogs

Walks every page of the `grp:frogs` query and keeps the fields the frog
pipeline needs. Page count comes from the first response, not a hardcoded
number.

In [ ]:
XENO_CANTO_QUERY_URL = 'https://xeno-canto.org/api/3/recordings'

# Free key from xeno-canto.org, kept out of the notebook since this is a public repo.
#   export XENO_CANTO_API_KEY="your-key-here"
xeno_canto_api_key = os.environ['XENO_CANTO_API_KEY']


def fetch_frog_recordings():
    """Page through the whole grp:frogs result set and return the raw records."""
    params = {'query': 'grp:frogs', 'key': xeno_canto_api_key, 'page': 1}
    first_page = requests.get(XENO_CANTO_QUERY_URL, params=params, timeout=30).json()

    # Page 1 is fetched on its own because its response is what tells us how many
    # pages there are.
    recordings = list(first_page['recordings'])
    total_pages = int(first_page['numPages'])
    for page_number in range(2, total_pages + 1):
        params['page'] = page_number
        page = requests.get(XENO_CANTO_QUERY_URL, params=params, timeout=30).json()
        recordings.extend(page['recordings'])
        time.sleep(random.uniform(2, 4))

    return recordings

In [ ]:
frog_records = pd.json_normalize(fetch_frog_recordings())

In [ ]:
frog_df = frog_records[
    ['gen', 'sp', 'cnt', 'loc', 'lat', 'lon', 'type', 'file', 'sono.med']
].copy()
frog_df.columns = [
    'Genus', 'Species', 'Country', 'Location',
    'Latitude', 'Longitude', 'Call Type', 'File', 'Spectrogram',
]

In [ ]:
%store frog_df
frog_df

## Downloading media

Downloads whatever each row references--spectrogram, audio, map--into
`~/Discrete_Signals/<Taxon>/<Species>/`, named so a species with multiple
recordings doesn't overwrite itself. Re-running just overwrites everything
fresh.

In [ ]:
def file_id_from_url(url):
    """The filename stem of a URL, used as a per-recording id."""
    return os.path.splitext(os.path.basename(str(url)))[0]


def download_asset(session, url, dest_path):
    """Download one file, skipping the HTML error pages SINA sometimes serves in
    place of a missing asset."""
    try:
        response = session.get(url, timeout=20)
        response.raise_for_status()
    except requests.RequestException as error:
        print(f'  download failed: {url}\n  {error}')
        return

    # SINA answers a missing asset with a 200 + an HTML error page, so check the
    # content type and sniff the first bytes rather than trusting the status code.
    content_type = response.headers.get('Content-Type', '').lower()
    if 'text/html' in content_type or response.content[:9].lower().startswith((b'<!doctype', b'<html')):
        print(f'  skipped (HTML, not an asset): {url}')
        return

    dest_path.write_bytes(response.content)


def _extension(url, allowed, default):
    """Lower-cased file extension of ``url`` if it's one we expect, else ``default``."""
    extension = os.path.splitext(str(url))[1].lower()
    return extension if extension in allowed else default

In [ ]:
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.gif', '.webp')
AUDIO_EXTENSIONS = ('.mp3', '.wav', '.ogg')


def download_sina_assets(df, taxon_folder_name):
    """Download the spectrogram, audio, and map for every row of a SINA
    DataFrame (cricket_df / katydid_df)."""
    session = make_session()
    taxon_dir = DISCRETE_SIGNALS_DIR / taxon_folder_name

    for _, row in df.iterrows():
        spectrogram_url, audio_url, map_url = row.get('Spectrogram'), row.get('Audio_Link'), row.get('Map')
        if pd.isna(spectrogram_url) and pd.isna(audio_url) and pd.isna(map_url):
            continue

        species = str(row['Species']).replace(' ', '_')
        species_dir = taxon_dir / species
        species_dir.mkdir(parents=True, exist_ok=True)

        # Prefer the spectrogram's id for the whole recording, so its audio file
        # is named to match rather than by its own (often generic) URL stem.
        recording_id = None
        if pd.notna(spectrogram_url) and spectrogram_url:
            recording_id = file_id_from_url(spectrogram_url)
            extension = _extension(spectrogram_url, IMAGE_EXTENSIONS, '.gif')
            download_asset(session, spectrogram_url,
                           species_dir / f'{species}_spectrogram_{recording_id}{extension}')

        if pd.notna(audio_url) and audio_url:
            audio_url = str(audio_url).strip()
            audio_id = recording_id or file_id_from_url(audio_url)
            extension = _extension(audio_url, AUDIO_EXTENSIONS, '.wav')
            download_asset(session, audio_url,
                           species_dir / f'{species}_audio_{audio_id}{extension}')

        if pd.notna(map_url) and map_url:
            extension = _extension(map_url, IMAGE_EXTENSIONS, '.gif')
            download_asset(session, map_url,
                           species_dir / f'{species}_map_{file_id_from_url(map_url)}{extension}')

In [ ]:
download_sina_assets(cricket_df, 'Crickets')

In [ ]:
download_sina_assets(katydid_df, 'Katydids')

### Frogs

No spectrogram id to key off of here, so frog audio is just numbered
sequentially per species (`_1`, `_2`, ...)--that number becomes `File_ID`
downstream. Re-running adds more files rather than replacing them, so delete
`~/Discrete_Signals/Frogs/` first if you want a clean re-scrape.

In [ ]:
def download_frog_audio(frog_df):
    """Download each frog recording's audio into
    ``~/Discrete_Signals/Frogs/<Genus>_<species>/<species>_audio_<n>.<ext>``."""
    session = make_session()
    frogs_dir = DISCRETE_SIGNALS_DIR / 'Frogs'
    total = len(frog_df)

    for position, (_, row) in enumerate(frog_df.iterrows(), start=1):
        audio_url = row.get('File')
        if pd.isna(audio_url):
            continue

        genus = str(row['Genus'])
        species = str(row['Species']).strip("'")
        species_dir = frogs_dir / f'{genus}_{species}'
        species_dir.mkdir(parents=True, exist_ok=True)

        # Xeno-Canto rows carry no stable per-file id we use downstream, so files
        # are numbered by how many already sit in the folder.
        extension = _extension(audio_url, AUDIO_EXTENSIONS, '.mp3')
        next_number = len(list(species_dir.glob(f'{species}_audio_*'))) + 1
        dest_path = species_dir / f'{species}_audio_{next_number}{extension}'

        try:
            response = session.get(audio_url, timeout=20)
            response.raise_for_status()
            dest_path.write_bytes(response.content)
            print(f'[{position}/{total}] saved {dest_path.name}')
        except requests.RequestException as error:
            print(f'[{position}/{total}] failed {audio_url}\n  {error}')

        time.sleep(random.uniform(1, 2))

In [ ]:
download_frog_audio(frog_df)